In [1]:
from langchain.tools import DuckDuckGoSearchResults

search = DuckDuckGoSearchResults()
search.run("Nico")

"[snippet: Nico was a German singer, model, and actress who collaborated with The Velvet Underground and other avant-garde artists. This article showcases 20 of her best songs, from her haunting covers of Dylan and The Doors to her own dark and hypnotic compositions., title: Best Nico Songs: 20 Enigmatic Classics From The ... - uDiscoverMusic, link: https://www.udiscovermusic.com/stories/best-nico-songs-and-velvet-underground/], [snippet: Tennessee's Nico Iamaleava (8) during Tennessee football's first fall practice, in Knoxville, Tenn., Wednesday, July 31, 2024. / Caitie McMekin/News Sentinel / USA TODAY NETWORK, title: Tennessee Volunteers Legend Says Nico Iamaleava is 'For Real', link: https://www.si.com/college/tennessee/football/tennessee-volunteer-legend-says-nico-iamaleava-is-for-real-01j5k4cm01pb], [snippet: Explore the enigmatic life of Nico, the iconic chanteuse of The Velvet Underground, in this captivating video journey. From her early days as a fashion model..., title: The

In [15]:
import os
import requests
from typing import Type
from langchain.chat_models import ChatOpenAI
from langchain.tools import BaseTool
from pydantic import BaseModel, Field
from langchain.agents import initialize_agent, AgentType
from langchain.utilities import DuckDuckGoSearchAPIWrapper

llm = ChatOpenAI(temperature=0.1, model_name="gpt-3.5-turbo-1106")

alpha_vantage_api_key = os.environ.get("ALPHA_VANTAGE_API_KEY")


class StockMarketSymbolSearchToolArgsSchema(BaseModel):
    query: str = Field(
        description="The query you will search for.Example query: Stock Market Symbol for Apple Company"
    )


class StockMarketSymbolSearchTool(BaseTool):
    name = "StockMarketSymbolSearchTool"
    description = """
    Use this tool to find the stock market symbol for a company.
    It takes a query as an argument.
    
    """
    args_schema: Type[
        StockMarketSymbolSearchToolArgsSchema
    ] = StockMarketSymbolSearchToolArgsSchema

    def _run(self, query):
        ddg = DuckDuckGoSearchAPIWrapper()
        return ddg.run(query)


class CompanyOverviewArgsSchema(BaseModel):
    symbol: str = Field(
        description="Stock symbol of the company.Example: AAPL,TSLA",
    )


class CompanyOverviewTool(BaseTool):
    name = "CompanyOverview"
    description = """
    Use this to get an overview of the financials of the company.
    You should enter a stock symbol.
    """
    args_schema: Type[CompanyOverviewArgsSchema] = CompanyOverviewArgsSchema

    def _run(self, symbol):
        r = requests.get(
            f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={alpha_vantage_api_key}"
        )
        return r.json()


class CompanyIncomeStatementTool(BaseTool):
    name = "CompanyIncomeStatement"
    description = """
    Use this to get the income statement of a company.
    You should enter a stock symbol.
    """
    args_schema: Type[CompanyOverviewArgsSchema] = CompanyOverviewArgsSchema

    def _run(self, symbol):
        r = requests.get(
            f"https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={symbol}&apikey={alpha_vantage_api_key}"
        )
        return r.json()["annualReports"]


class CompanyStockPerformanceTool(BaseTool):
    name = "CompanyStockPerformance"
    description = """
    Use this to get the weekly performance of a company stock.
    You should enter a stock symbol.
    """
    args_schema: Type[CompanyOverviewArgsSchema] = CompanyOverviewArgsSchema

    def _run(self, symbol):
        r = requests.get(
            f"https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY&symbol={symbol}&apikey={alpha_vantage_api_key}"
        )
        return r.json()


agent = initialize_agent(
    llm=llm,
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    handle_parsing_errors=True,
    tools=[
        StockMarketSymbolSearchTool(),
        CompanyOverviewTool(),
        CompanyIncomeStatementTool(),
        CompanyStockPerformanceTool(),
    ],
)

prompt = "Give me financial information on Cloudflare's stock, considering its financials, income statements and stock performance help me analyze if it's a potential good investment."

agent.invoke(prompt)



> Entering new AgentExecutor chain...

Invoking: `StockMarketSymboleSearchTool` with `{'query': 'Cloudflare'}`


[snippet: Cloudflare Stream is an end-to-end solution for video encoding, storage, delivery, and playback, focused on simplifying all aspects of video for developers. Newly uploaded or ingested portrait videos will now automatically be processed in full HD quality ..., title: The Cloudflare Blog, link: https://blog.cloudflare.com/], [snippet: Cloudflare's connectivity cloud protects entire corporate networks, helps customers build Internet-scale applications efficiently, accelerates any website or Internet application, wards off DDoS attacks, keeps hackers at bay, and can help you on your journey to Zero Trust. Visit 1.1.1.1 from any device to get started with our free app that makes your Internet faster and safer., title: Cloudflare is now powering Microsoft Edge Secure Network, link: https://blog.cloudflare.com/cloudflare-now-powering-microsoft-edge-secure-network/], [sn

{'input': "Give me financial information on Cloudflare's stock, considering its financials, income statements and stock performance help me analyze if it's a potential good investment.",
 'output': "### Cloudflare Inc (NET) Financial Information:\n\n- **Sector:** Technology\n- **Industry:** Services-Prepackaged Software\n- **Market Capitalization:** $28,149,494,000\n- **Revenue (TTM):** $1,477,674,000\n- **Gross Profit (TTM):** $742,631,000\n- **EBITDA:** -$48,084,000\n- **EPS (TTM):** -$0.3\n- **Profit Margin:** -6.9%\n- **Operating Margin (TTM):** -8.65%\n- **Return on Assets (TTM):** -3.59%\n- **Return on Equity (TTM):** -13.4%\n- **Quarterly Revenue Growth (YoY):** 0.3%\n- **Analyst Target Price:** $92.67\n- **52-Week High:** $116\n- **52-Week Low:** $53.88\n- **50-Day Moving Average:** $79.05\n- **200-Day Moving Average:** $82.32\n- **Shares Outstanding:** 303,676,000\n- **Beta:** 1.1\n\n### Analysis:\n- Cloudflare Inc has shown a positive quarterly revenue growth year over year.\